In [21]:
#importing needed libraries
import pandas as pd
import re

### **Cleaning the Qiwa website data file (qiwa_data.csv)**


* Handled the single column Issue by having the code read the file line by line to correctly separate the URL and the content.

* Consolidated content by grouping all the scattered text belonging to the same URL into a single cell.

* Processed canceled articles by identifing the URL for the chapter where all articles have been canceled, and replaced with a note stating that the entire chapter is void.

* Removed duplication that occurred during the web scraping process.

* Saved the clean data to a new CSV file (`qiwa_data_cleaned.csv`).

In [22]:
def clean_qiwa_data(input_file='qiwa_data.csv', output_file='qiwa_data_cleaned.csv'):
    try:
        with open(input_file, 'r', encoding='utf-8-sig') as f:
            lines = f.readlines()

        records = []
        current_url = None

        if 'URL,Content' in lines[0]:
            lines = lines[1:]

        for line in lines:
            line = line.strip()
            if not line:
                continue

            if line.startswith('http'):
                parts = line.split(',', 1)
                current_url = parts[0].strip()
                initial_content = parts[1].strip('"').strip() if len(parts) > 1 else ''
                if initial_content:
                    records.append({'URL': current_url, 'Content': initial_content})
            elif current_url:
                content = line.strip('"').strip()
                records.append({'URL': current_url, 'Content': content})

        if not records:
            print(f"❌ لم تتم معالجة أي بيانات من '{input_file}'.")
            return

        processed_df = pd.DataFrame(records)
        cleaned_df = processed_df.groupby('URL')['Content'].apply(lambda x: '\n'.join(x)).reset_index()

        cancelled_url = "https://qiwa.sa/ar/labor-law/contracts/commissions-settlement-labor-disputes"
        cancellation_note = "الباب الثالث عشر: هيئات تسوية الخلافات العمالية (ملاحظة: جميع المواد في هذا الباب، من المادة 210 إلى 228، قد تم إلغاؤها)."
        cleaned_df.loc[cleaned_df['URL'] == cancelled_url, 'Content'] = cancellation_note

        def remove_major_duplication(text):
            lines = text.splitlines()
            if len(lines) > 1 and len(lines) % 2 == 0:
                half = len(lines) // 2
                if lines[:half] == lines[half:]:
                    return '\n'.join(lines[:half])
            return text

        mask = cleaned_df['URL'] != cancelled_url
        cleaned_df.loc[mask, 'Content'] = cleaned_df.loc[mask, 'Content'].apply(remove_major_duplication)

        cleaned_df.to_csv(output_file, index=False, encoding='utf-8-sig')

        print(f"✅ Successfully cleaned '{input_file}' and saved to '{output_file}'.")

    except FileNotFoundError:
        print(f"❌ Error: The file '{input_file}' was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

clean_qiwa_data()

✅ Successfully cleaned 'qiwa_data.csv' and saved to 'qiwa_data_cleaned.csv'.


### **Cleaning the FAQ file (labor_law_faq.csv)**


* Unified text by processeing the text in both the "Question" and "Answer" columns.

* Removed line breaks and extra whitespace and replaced them with a single space.

* Trims edges by removeing any whitespace from the beginning or end of each question and answer.

* Saved the clean data to a new CSV file (`labor_law_faq_cleaned.csv`).

In [23]:
def clean_faq_data(input_file='labor_law_faq.csv', output_file='labor_law_faq_cleaned.csv'):
    """
    Cleans the FAQ data and saves it in the standard comma-separated format.
    """
    try:
        df = pd.read_csv(input_file)
        if 'Question' not in df.columns or 'Answer' not in df.columns:
            print("❌ Error: CSV must contain 'Question' and 'Answer' columns.")
            return

        for col in ['Question', 'Answer']:
            df[col] = df[col].astype(str).str.replace(r'[\r\n]+', ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()

        # Saving as a standard CSV (comma-separated)
        df.to_csv(output_file, index=False, encoding='utf-8-sig')

        print(f"✅ Successfully cleaned '{input_file}' and saved to '{output_file}'.")

    except FileNotFoundError:
        print(f"❌ Error: The file '{input_file}' was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

# Run the cleaning function
clean_faq_data()

✅ Successfully cleaned 'labor_law_faq.csv' and saved to 'labor_law_faq_cleaned.csv'.
